<a href="https://www.kaggle.com/code/avikdas567/rental-product-recommendation-system?scriptVersionId=295947437" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import gc
import warnings
import re

# Suppress warnings
warnings.filterwarnings('ignore')

# ==========================================
# CONFIGURATION
# ==========================================
BASE_PATH = '/kaggle/input/rental-product-recommendation-system/'
FILES = {
    'hits': BASE_PATH + 'metrika_hits_test.csv',
    'visits': BASE_PATH + 'metrika_visits_test.csv',
    'products': BASE_PATH + 'new_site_products.csv',
    'orders': BASE_PATH + 'new_site_orders.csv'
}

# ==========================================
# 1. LOAD & MAP PRODUCT CATALOG
# ==========================================
print("Loading Product Catalog...")
products_df = pd.read_csv(FILES['products'], usecols=['id', 'slug'])

# Create mappings
slug_to_id = dict(zip(products_df['slug'], products_df['id']))
all_product_ids = products_df['id'].unique()

print(f"Catalog Size: {len(all_product_ids)} items.")
del products_df
gc.collect()

# ==========================================
# 2. BUILD ORDER SIMILARITY MATRIX (Buy-to-Buy)
# ==========================================
print("Building Order Co-occurrence Matrix...")
orders_df = pd.read_csv(FILES['orders'], usecols=['product_id', 'number'])

# 1. Global Popularity (Backup strategy)
pop_counts = orders_df['product_id'].value_counts()
global_top_6 = pop_counts.head(6).index.tolist()

# 2. Build Matrix
# We use a nested dictionary: matrix[item_a][item_b] = score
order_matrix = defaultdict(float)
order_counts = defaultdict(int)

# Group by order
order_groups = orders_df.groupby('number')['product_id'].apply(list)

for items in order_groups:
    if len(items) > 1:
        # Normalize weight by order size (smaller orders = stronger link)
        weight = 1.0 / (len(items) - 1)
        for i in items:
            order_counts[i] += 1
            for j in items:
                if i != j:
                    # Link i -> j
                    order_matrix[(i, j)] += weight

print(f"Processed {len(order_groups)} orders.")
del orders_df, order_groups
gc.collect()

# ==========================================
# 3. BUILD SESSION SIMILARITY MATRIX (View-to-View)
# ==========================================
print("Processing Test Sessions for Co-visitation...")

# A. Map Hits (WatchID -> ProductID)
hits_df = pd.read_csv(FILES['hits'], usecols=['watch_id', 'slug'], dtype={'watch_id': str})
hits_df['product_id'] = hits_df['slug'].map(slug_to_id)
hits_df = hits_df.dropna(subset=['product_id']) # Drop non-product hits

# Lookup: watch_id -> product_id
watch_id_to_product = dict(zip(hits_df['watch_id'], hits_df['product_id'].astype(int)))

del hits_df
gc.collect()

# B. Process Visits (The Sessions)
visits_df = pd.read_csv(FILES['visits'], usecols=['visit_id', 'watch_ids'])
view_matrix = defaultdict(float)

# Regex to parse the list string faster than ast.literal_eval
pattern = re.compile(r'\d+') 

print("Computing Session Co-occurrence...")
session_histories = {} # Store for prediction step

for row in visits_df.itertuples(index=False):
    visit_id = row[0]
    raw_ids = pattern.findall(str(row[1]))
    
    # Resolve to product IDs
    history = []
    for wid in raw_ids:
        if wid in watch_id_to_product:
            history.append(watch_id_to_product[wid])
    
    session_histories[visit_id] = history
    
    # Update View Matrix
    # We weight pairs by proximity in the session
    if len(history) > 1:
        for i in range(len(history)):
            item_a = history[i]
            # Look at a window of neighbors (e.g., +/- 2 steps)
            start = max(0, i-2)
            end = min(len(history), i+3)
            
            for j in range(start, end):
                if i != j:
                    item_b = history[j]
                    # Items closer in time get higher weight
                    diff = abs(i - j)
                    weight = 1.0 / diff # 1.0 for immediate neighbor, 0.5 for 1-hop away
                    view_matrix[(item_a, item_b)] += weight

print("Matrices Built.")

# ==========================================
# 4. GENERATE SCORES & PREDICT
# ==========================================
print("Generating predictions...")

preds = []
# Weights for our logic
W_ORDERS = 3.0  # Buying together is a very strong signal
W_VIEWS = 1.0   # Viewing together is a medium signal
W_RECENT = 1.5  # Bias towards items related to the VERY LAST item viewed

for visit_id in visits_df['visit_id']:
    candidates = defaultdict(float)
    history = session_histories.get(visit_id, [])
    
    if history:
        # We look at the last 3 items in history to form context
        recent_history = history[-3:] 
        
        for i, item in enumerate(reversed(recent_history)):
            # "i" is distance from end (0 = last item, 1 = second to last...)
            decay = 1.0 / (i + 1) # 1.0, 0.5, 0.33
            
            pass 

        pass

print("Optimizing Lookup Structures...")
fast_order_graph = defaultdict(dict)
for (i, j), w in order_matrix.items():
    fast_order_graph[i][j] = w

fast_view_graph = defaultdict(dict)
for (i, j), w in view_matrix.items():
    fast_view_graph[i][j] = w
    
del order_matrix, view_matrix
gc.collect()

# --- FINAL PREDICTION LOOP ---
final_submission_data = []

for visit_id in visits_df['visit_id']:
    scores = defaultdict(float)
    history = session_histories.get(visit_id, [])
    seen_items = set(history)
    
    if history:
        # Focus on the last item (Strongest Intent)
        last_item = history[-1]
        
        # 1. Get neighbors from Order Graph (Strongest)
        if last_item in fast_order_graph:
            for cand, w in fast_order_graph[last_item].items():
                if cand not in seen_items:
                    scores[cand] += w * W_ORDERS
        
        # 2. Get neighbors from View Graph (Discovery)
        if last_item in fast_view_graph:
            for cand, w in fast_view_graph[last_item].items():
                if cand not in seen_items:
                    scores[cand] += w * W_VIEWS

        # 3. Context Boost: Look at 2nd to last item too
        if len(history) > 1:
            second_last = history[-2]
            if second_last in fast_order_graph:
                for cand, w in fast_order_graph[second_last].items():
                    if cand not in seen_items:
                        scores[cand] += (w * W_ORDERS) * 0.5 # Decay weight
    
    # Sort candidates by score
    sorted_candidates = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    top_candidates = [x[0] for x in sorted_candidates[:6]]
    
    # FILLER: If we don't have 6 yet, fill with Global Bestsellers
    if len(top_candidates) < 6:
        for item in global_top_6:
            if item not in top_candidates and item not in seen_items:
                top_candidates.append(item)
            if len(top_candidates) == 6:
                break
    
    # Convert to string space-delimited
    pred_str = " ".join(map(str, top_candidates))
    final_submission_data.append(pred_str)

# ==========================================
# 5. WRITE SUBMISSION
# ==========================================
visits_df['product_ids'] = final_submission_data
submission = visits_df[['visit_id', 'product_ids']]
submission.to_csv('submission.csv', index=False)

print("Success! 'submission.csv' created.")
display(submission.head())

Loading Product Catalog...
Catalog Size: 668 items.
Building Order Co-occurrence Matrix...
Processed 2259 orders.
Processing Test Sessions for Co-visitation...
Computing Session Co-occurrence...
Matrices Built.
Generating predictions...
Optimizing Lookup Structures...
Success! 'submission.csv' created.


,visit_id,product_ids
0,6034151852304141635,463480429 463480693 495400618 463480694 495401...
1,0473312698303184850,463480429 463480693 495400618 463480694 495401...
2,0140437049255950634,463480255 463480256 463480697 463480236 463480...
3,4595976301611479438,463480429 463480693 495400618 463480694 495401...
4,1835293676913553579,495277589 463480429 463480225 463480224 463480...
